In [1]:
%cd ..

/home/bhchen/LearnKalmanGain


In [2]:
import os
import glob
import csv
import math
from collections import defaultdict
from PIL import Image
from pathlib import Path
import re
from typing import List, Tuple, Dict, Optional, Any

import matplotlib.pyplot as plt


SNAPSHOT_PERCENTILES = (25, 50, 75, 100)


def _build_snapshot_specs(step_values: List[Any], percentiles: Tuple[int, ...] = SNAPSHOT_PERCENTILES):
    """Select fixed-percentile frames using the same ordering as the GIF."""
    if len(step_values) == 0:
        return []

    specs = []
    total = len(step_values)
    for pct in percentiles:
        idx = min(total - 1, max(0, math.ceil(total * (pct / 100.0)) - 1))
        specs.append({
            "percentile": pct,
            "index": idx,
            "step": step_values[idx],
        })
    return specs


def save_selected_gif_frames(
    frames: List[Image.Image],
    step_values: List[Any],
    gif_path: str,
    percentiles: Tuple[int, ...] = SNAPSHOT_PERCENTILES,
) -> List[Path]:
    """Save fixed snapshots whose contents exactly match selected GIF frames."""
    if len(frames) == 0 or len(step_values) == 0:
        return []
    if len(frames) != len(step_values):
        raise ValueError("frames and step_values must have the same length.")

    gif_file = Path(gif_path)
    saved_paths: List[Path] = []

    for spec in _build_snapshot_specs(step_values, percentiles):
        out_path = gif_file.with_name(
            f"{gif_file.stem}_step{spec['percentile']}pct_timestep{spec['step']}.png"
        )
        frames[spec["index"]].save(out_path)
        saved_paths.append(out_path)
        print(
            f"Saved frame snapshot: {out_path} "
            f"(percentile={spec['percentile']}%, timestep={spec['step']})"
        )

    return saved_paths


def create_gif_robust(image_folder, file_pattern, gif_path, duration=100, max_value=None):
    """
    Function:
        Creates a GIF from images matching a pattern with a numerical wildcard.
        This version is robust and does not depend on any keywords like 'timestep'.
        It identifies the number by what the '*' in the pattern matches.
    Input:
        image_folder (str): The path to the folder containing the images.
        file_pattern (str): Filename pattern with one '*' as a wildcard for a number.
                            Example: "frame_*_render.png"
        gif_path (str): The path to save the output GIF file.
        duration (int): Duration (in milliseconds) for each frame.
        max_value (int, optional): The maximum value of the wildcard part to include.
                                If None, all matched images will be used.
    Output:
        None
    """
    if file_pattern.count('*') != 1:
        print("Error: The file_pattern must contain exactly one '*' wildcard.")
        return

    # Dynamically determine the prefix and suffix from the pattern
    prefix, suffix = file_pattern.split('*')

    search_path = os.path.join(image_folder, file_pattern)
    all_filenames = glob.glob(search_path)

    if not all_filenames:
        print(f"Error: No images found in '{image_folder}' matching '{file_pattern}'.")
        return

    files_with_values = []
    for f_path in all_filenames:
        basename = os.path.basename(f_path)
        # Extract the part of the filename that corresponds to the wildcard
        if basename.startswith(prefix) and basename.endswith(suffix):
            try:
                # Remove prefix and suffix to get the numerical part
                value_str = basename[len(prefix):-len(suffix)]
                value = int(value_str)
                files_with_values.append({'value': value, 'path': f_path})
            except (ValueError, IndexError):
                # Ignore files where the wildcard part is not a valid integer
                print(f"Warning: Could not extract a valid number from '{basename}'. Skipping.")
                continue

    if not files_with_values:
        print("Error: Found matching files, but could not extract numbers from any of them.")
        return

    # Sort the files based on the extracted numerical value
    files_with_values.sort(key=lambda item: item['value'])

    # Filter by max_value if provided
    if max_value is not None:
        print(f"Filtering frames to include values up to {max_value}.")
        files_with_values = [item for item in files_with_values if item['value'] <= max_value]

    if not files_with_values:
        print(f"Error: After filtering, no images remained with a value <= {max_value}.")
        return

    final_filenames = [item['path'] for item in files_with_values]
    step_values = [item['value'] for item in files_with_values]
    print(f"Creating GIF with {len(final_filenames)} frames...")

    images = []
    for fn in final_filenames:
        with Image.open(fn) as im:
            images.append(im.convert("RGB").copy())

    images[0].save(
        gif_path,
        save_all=True,
        append_images=images[1:],
        duration=duration,
        loop=0
    )
    save_selected_gif_frames(images, step_values, gif_path)
    for im in images:
        im.close()
    print(f"GIF saved successfully at: {gif_path}")


def pair_prior_post(
    image_directory: str,
    pattern: str,
    left_tag: str = "PRIOR",
    right_tag: str = "POST",
    concat_token: str = "PRIOR_POST",
    overwrite: bool = False,
) -> Tuple[int, List[Path]]:
    """
    Find PRIOR/POST image pairs for the same timestep and save a side-by-side concat
    (PRIOR on the left, POST on the right) in the same directory as the sources.
    """

    root = Path(image_directory)
    if not root.exists():
        raise FileNotFoundError(f"Directory not found: {root}")

    # --- Build regex safely ---
    escaped = re.escape(pattern)

    # Ensure we have timestep* to capture digits
    if r"timestep\*" not in escaped:
        raise ValueError("Pattern must contain 'timestep*' so the timestep can be captured.")
    escaped = escaped.replace(r"timestep\*", r"timestep(\d+)")

    # Replace the FIRST occurrence of the escaped right_tag with a capture group (left|right)
    escaped_right = re.escape(right_tag)
    escaped_left = re.escape(left_tag)

    if escaped_right not in escaped:
        raise ValueError(
            f"Could not find the tag '{right_tag}' inside the pattern. "
            f"Please ensure pattern includes it exactly once."
        )

    escaped = escaped.replace(escaped_right, f"({escaped_left}|{escaped_right})", 1)

    name_re = re.compile(rf"^{escaped}$")

    # --- Glob files: widen the tag position (POST -> *) to catch both PRIOR/POST ---
    glob_pattern = pattern.replace(right_tag, "*", 1)
    files = list(root.glob(glob_pattern))

    by_step: Dict[int, Dict[str, Path]] = {}
    for p in files:
        m = name_re.match(p.name)
        if not m:
            continue
        # Guard against malformed regex (should have 2 groups: timestep, tag)
        if m.lastindex is None or m.lastindex < 2:
            continue

        step = int(m.group(1))
        tag = m.group(2)
        by_step.setdefault(step, {})[tag] = p

    pairs = [(s, d.get(left_tag), d.get(right_tag)) for s, d in by_step.items()]
    # Keep only complete pairs
    pairs = [(s, a, b) for (s, a, b) in pairs if a is not None and b is not None]
    pairs.sort(key=lambda x: x[0])

    saved_paths: List[Path] = []

    def build_out_name(post_name: str) -> str:
        """Replace the tag segment with concat_token using the compiled regex spans."""
        m = name_re.match(post_name)
        if not m or m.lastindex is None or m.lastindex < 2:
            # Fallback: a simple first-occurrence replacement
            return post_name.replace(right_tag, concat_token, 1)
        a, b = m.span(2)
        return post_name[:a] + concat_token + post_name[b:]

    def concat_horiz(im_left: Image.Image, im_right: Image.Image) -> Image.Image:
        """Concatenate two images horizontally (left then right)."""
        if im_left.height != im_right.height:
            scale = im_left.height / im_right.height
            new_w = max(1, int(round(im_right.width * scale)))
            im_right = im_right.resize((new_w, im_left.height), Image.BICUBIC)
        w = im_left.width + im_right.width
        h = max(im_left.height, im_right.height)
        canvas = Image.new("RGB", (w, h), (255, 255, 255))
        canvas.paste(im_left, (0, 0))
        canvas.paste(im_right, (im_left.width, 0))
        return canvas

    created = 0
    for step, prior_path, post_path in pairs:
        out_name = build_out_name(post_path.name)
        out_path = prior_path.parent / out_name

        if out_path.exists() and not overwrite:
            continue

        with Image.open(prior_path) as im_prior, Image.open(post_path) as im_post:
            im_prior = im_prior.convert("RGB")
            im_post = im_post.convert("RGB")
            im_cat = concat_horiz(im_prior, im_post)
            im_cat.save(out_path)

        created += 1
        saved_paths.append(out_path)

    return created, saved_paths


def collect_prior_post_pairs(
    image_directory: str,
    pattern: str,
    left_tag: str = "PRIOR",
    right_tag: str = "POST",
) -> List[Tuple[int, Path, Path]]:
    """
    Collect matched PRIOR/POST pairs as (timestep, prior_path, post_path), sorted by timestep.
    """
    root = Path(image_directory)
    if not root.exists():
        raise FileNotFoundError(f"Directory not found: {root}")

    escaped = re.escape(pattern)
    if r"timestep\*" not in escaped:
        raise ValueError("Pattern must contain 'timestep*' so the timestep can be captured.")
    escaped = escaped.replace(r"timestep\*", r"timestep(\d+)")

    escaped_right = re.escape(right_tag)
    escaped_left = re.escape(left_tag)
    if escaped_right not in escaped:
        raise ValueError(
            f"Could not find the tag '{right_tag}' inside the pattern. "
            f"Please ensure pattern includes it exactly once."
        )
    escaped = escaped.replace(escaped_right, f"({escaped_left}|{escaped_right})", 1)
    name_re = re.compile(rf"^{escaped}$")

    glob_pattern = pattern.replace(right_tag, "*", 1)
    files = list(root.glob(glob_pattern))

    by_step: Dict[int, Dict[str, Path]] = {}
    for p in files:
        m = name_re.match(p.name)
        if not m:
            continue
        if m.lastindex is None or m.lastindex < 2:
            continue

        step = int(m.group(1))
        tag = m.group(2)
        by_step.setdefault(step, {})[tag] = p

    pairs = []
    for step, d in by_step.items():
        p_left = d.get(left_tag)
        p_right = d.get(right_tag)
        if p_left is not None and p_right is not None:
            pairs.append((step, p_left, p_right))

    pairs.sort(key=lambda x: x[0])
    return pairs


def create_prior_post_switch_gif(
    image_directory: str,
    pattern: str,
    gif_path: str,
    duration: int = 600,
    max_steps: int = 30,
    left_tag: str = "PRIOR",
    right_tag: str = "POST",
):
    """
    Build a slow GIF that alternates PRIOR -> POST for each timestep.
    Sequence: [prior_t1, post_t1, prior_t2, post_t2, ...]
    """
    pairs = collect_prior_post_pairs(
        image_directory=image_directory,
        pattern=pattern,
        left_tag=left_tag,
        right_tag=right_tag,
    )

    if len(pairs) == 0:
        print(f"Error: No PRIOR/POST pairs found for pattern '{pattern}'.")
        return

    if max_steps is not None and max_steps > 0:
        pairs = pairs[:max_steps]

    if len(pairs) == 0:
        print("Error: No frames left after applying max_steps filter.")
        return

    frames = []
    for _, prior_path, post_path in pairs:
        with Image.open(prior_path) as im_prior:
            frames.append(im_prior.convert("RGB").copy())
        with Image.open(post_path) as im_post:
            frames.append(im_post.convert("RGB").copy())

    if len(frames) == 0:
        print("Error: No frame images were loaded.")
        return

    frames[0].save(
        gif_path,
        save_all=True,
        append_images=frames[1:],
        duration=duration,
        loop=0,
    )

    for im in frames:
        im.close()

    print(
        f"Switch GIF saved successfully at: {gif_path} "
        f"(steps={len(pairs)}, frames={2 * len(pairs)}, duration={duration}ms)"
    )


def _extract_l63_multiview_fields(name: str):
    sigma_m = re.search(r"sigma_y([^_]+)", name)
    step_m = re.search(r"timestep(\d+)", name)
    mode_m = re.search(r"_(prior|post)_", name, flags=re.IGNORECASE)
    view_m = re.search(r"_(xy|yz|xz)\.png$", name, flags=re.IGNORECASE)
    if sigma_m is None or step_m is None or mode_m is None or view_m is None:
        return None
    return {
        "sigma": sigma_m.group(1),
        "step": int(step_m.group(1)),
        "mode": mode_m.group(1).lower(),
        "view": view_m.group(1).lower(),
    }


def create_l63_prior_post_projection_gif(
    image_directory: str,
    post_xy_pattern: str,
    gif_path: str,
    duration: int = 250,
    max_value: Optional[int] = None,
    sigma_value: Optional[str] = None,
):
    """
    Build Lorenz63 multiview GIF where each frame is a 2x3 grid:
      top row: prior (xy, yz, xz)
      bottom row: post (xy, yz, xz)

    `post_xy_pattern` is a glob pattern for post-xy images, e.g.
      sigma_y*_..._timestep*_..._post_adaptive_xy.png
    """
    root = Path(image_directory)
    if not root.exists():
        raise FileNotFoundError(f"Directory not found: {root}")

    if "_post_" not in post_xy_pattern:
        raise ValueError("Pattern must include '_post_' for mode expansion.")
    if not post_xy_pattern.endswith("_xy.png"):
        raise ValueError("Pattern must end with '_xy.png' for xy/yz/xz expansion.")

    search_pattern = post_xy_pattern.replace("_post_", "_*_", 1).replace("_xy.png", "_*.png", 1)
    candidates = list(root.glob(search_pattern))
    if len(candidates) == 0:
        print(f"Error: No L63 multiview files found for '{search_pattern}'.")
        return None

    grouped: Dict[str, Dict[int, Dict[str, Dict[str, Path]]]] = defaultdict(
        lambda: defaultdict(lambda: defaultdict(dict))
    )
    for p in candidates:
        fields = _extract_l63_multiview_fields(p.name)
        if fields is None:
            continue
        grouped[fields["sigma"]][fields["step"]][fields["mode"]][fields["view"]] = p

    required_views = ("xy", "yz", "xz")

    def complete_steps_for_sigma(sigma_key: str) -> List[int]:
        steps = []
        for step, modes in grouped[sigma_key].items():
            prior_views = modes.get("prior", {})
            post_views = modes.get("post", {})
            if all(v in prior_views for v in required_views) and all(v in post_views for v in required_views):
                steps.append(step)
        steps.sort()
        return steps

    sigma_to_steps = {}
    for sigma_key in grouped.keys():
        steps = complete_steps_for_sigma(sigma_key)
        if len(steps) > 0:
            sigma_to_steps[sigma_key] = steps

    if len(sigma_to_steps) == 0:
        print("Error: Found files, but no complete (prior/post x xy/yz/xz) timestep groups.")
        return None

    if sigma_value is not None and sigma_value in sigma_to_steps:
        selected_sigma = sigma_value
    else:
        selected_sigma = sorted(
            sigma_to_steps.keys(),
            key=lambda k: (len(sigma_to_steps[k]), k),
            reverse=True,
        )[0]
        if sigma_value is not None and sigma_value not in sigma_to_steps:
            print(
                f"Warning: sigma_y{sigma_value} not found with complete groups. "
                f"Fallback to sigma_y{selected_sigma}."
            )

    steps = sigma_to_steps[selected_sigma]
    if max_value is not None:
        steps = [s for s in steps if s <= max_value]

    if len(steps) == 0:
        print(f"Error: No frames left after max_value filter for sigma_y{selected_sigma}.")
        return None

    frames: List[Image.Image] = []
    views_order = ["xy", "yz", "xz"]

    for step in steps:
        paths = grouped[selected_sigma][step]
        sample_path = paths["prior"]["xy"]
        with Image.open(sample_path) as im0:
            tile_size = im0.convert("RGB").size

        tile_w, tile_h = tile_size
        canvas = Image.new("RGB", (tile_w * 3, tile_h * 2), (255, 255, 255))

        for row, mode in enumerate(["prior", "post"]):
            for col, view in enumerate(views_order):
                img_path = paths[mode][view]
                with Image.open(img_path) as im:
                    tile = im.convert("RGB")
                    if tile.size != tile_size:
                        tile = tile.resize(tile_size, Image.BICUBIC)
                    canvas.paste(tile, (col * tile_w, row * tile_h))

        frames.append(canvas)

    frames[0].save(
        gif_path,
        save_all=True,
        append_images=frames[1:],
        duration=duration,
        loop=0,
    )
    save_selected_gif_frames(frames, steps, gif_path)

    for im in frames:
        im.close()

    print(
        f"L63 multiview GIF saved at: {gif_path} "
        f"(sigma_y={selected_sigma}, steps={len(steps)}, duration={duration}ms)"
    )
    return {
        "sigma": selected_sigma,
        "num_steps": len(steps),
        "steps": steps,
    }


def _load_distance_records_from_pt(distance_pt_path: Path) -> List[Dict[str, Any]]:
    import torch

    payload = torch.load(distance_pt_path, map_location="cpu")
    if isinstance(payload, list):
        return [x for x in payload if isinstance(x, dict)]

    if isinstance(payload, dict):
        for key in ["records", "distance_records", "data", "detail"]:
            val = payload.get(key)
            if isinstance(val, list):
                return [x for x in val if isinstance(x, dict)]

    raise ValueError(f"Unsupported PT payload type: {type(payload)}")


def _load_distance_records_from_csv(distance_csv_path: Path) -> List[Dict[str, Any]]:
    records = []
    with open(distance_csv_path, "r", newline="") as f_csv:
        reader = csv.DictReader(f_csv)
        for row in reader:
            try:
                step = int(row.get("step", ""))
            except Exception:
                continue
            rec = {"step": step}
            for key in [
                "prior_swd_ratio",
                "post_swd_ratio",
                "prior_swd_data",
                "post_swd_data",
                "prior_swd_baseline",
                "post_swd_baseline",
            ]:
                try:
                    rec[key] = float(row.get(key, float("nan")))
                except Exception:
                    rec[key] = float("nan")
            records.append(rec)
    return records


def plot_swd_vs_step(
    image_directory: str,
    distance_pt_pattern: str,
    output_name: Optional[str] = None,
):
    """
    Find PT detail file by glob pattern, load SWD records, and save SWD-vs-step figure.
    If PT loading fails, it falls back to the sibling CSV detail file.
    """
    root = Path(image_directory)
    if not root.exists():
        raise FileNotFoundError(f"Directory not found: {root}")

    pt_files = sorted(root.glob(distance_pt_pattern))
    if len(pt_files) == 0:
        print(f"No distance detail PT found in '{image_directory}' for pattern '{distance_pt_pattern}'.")
        return None

    pt_path = sorted(pt_files, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    records: List[Dict[str, Any]] = []
    source = f"pt:{pt_path.name}"

    try:
        records = _load_distance_records_from_pt(pt_path)
    except Exception as exc:
        csv_path = pt_path.with_suffix(".csv")
        if not csv_path.exists():
            print(f"PT load failed ({exc}) and CSV fallback not found: {csv_path}")
            return None
        records = _load_distance_records_from_csv(csv_path)
        source = f"csv:{csv_path.name}"
        print(f"Warning: PT load failed ({exc}); using CSV fallback: {csv_path.name}")

    if len(records) == 0:
        print(f"No SWD records available from {source}.")
        return None

    step_groups: Dict[int, Dict[str, List[float]]] = defaultdict(lambda: {"prior": [], "post": []})
    for rec in records:
        try:
            step = int(rec.get("step", -1))
        except Exception:
            continue
        if step < 0:
            continue

        for src_key, dst_key in [("prior_swd_ratio", "prior"), ("post_swd_ratio", "post")]:
            try:
                value = float(rec.get(src_key, float("nan")))
            except Exception:
                value = float("nan")
            if math.isfinite(value):
                step_groups[step][dst_key].append(value)

    if len(step_groups) == 0:
        print(f"No finite SWD-ratio values found in {source}.")
        return None

    steps = sorted(step_groups.keys())
    prior_mean = [
        (sum(step_groups[s]["prior"]) / len(step_groups[s]["prior"])) if len(step_groups[s]["prior"]) > 0 else float("nan")
        for s in steps
    ]
    post_mean = [
        (sum(step_groups[s]["post"]) / len(step_groups[s]["post"])) if len(step_groups[s]["post"]) > 0 else float("nan")
        for s in steps
    ]

    if output_name is None:
        output_name = f"{pt_path.stem}_swd_vs_step.png"
    output_path = root / output_name

    plt.figure(figsize=(8.8, 4.8))
    plt.plot(steps, prior_mean, color="tab:blue", linewidth=2.0, label="Prior SWD ratio mean")
    plt.plot(steps, post_mean, color="tab:red", linewidth=2.0, label="Posterior SWD ratio mean")
    plt.xlabel("Step")
    plt.ylabel("SWD ratio")
    plt.title(f"SWD vs step ({source})")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()

    print(f"SWD-vs-step figure saved at: {output_path}")
    return output_path



In [3]:
# Unified animation jobs across multiple datasets
ANIMATION_JOBS = {
    "doubling1d_ring_g0_post": {
        "dataset": "doubling1d",
        "image_directory": "save/doubling1d_pf_vis",
        "pattern": "sigma_y0.2_batch64_len200_pfN1000000_timestep*_42_g0_POST_0_ring.png",
        "frame_duration_ms": 200,
        "max_value": 200,
        "output_name": "doubling1d_traj0_200ms_200timesteps.gif",
        "concat_prior_post": False,
    },
    "doubling1d_ring_g0_prior_post": {
        "dataset": "doubling1d",
        "image_directory": "save/doubling1d_pf_vis",
        "pattern": "sigma_y0.2_batch64_len200_pfN1000000_timestep*_42_g0_POST_0_ring.png",
        "frame_duration_ms": 500,
        "max_value": 30,
        "output_name": "doubling1d_traj0_prior_post_200ms_200timesteps.gif",
        "concat_prior_post": True,
    },
    "complex2d_ring_g0_post": {
        "dataset": "complex2d",
        "image_directory": "save/complex2d_pf_vis",
        "pattern": "sigma_y0.2_batch64_len200_pfN1000000_timestep*_42_g0_POST_0_ring.png",
        "frame_duration_ms": 200,
        "max_value": 200,
        "output_name": "complex2d_traj0_200ms_200timesteps.gif",
        "concat_prior_post": False,
    },
    "complex2d_ring_g0_prior_post": {
        "dataset": "complex2d",
        "image_directory": "save/complex2d_pf_vis",
        "pattern": "sigma_y0.2_batch64_len200_pfN1000000_timestep*_42_g0_POST_0_ring.png",
        "frame_duration_ms": 200,
        "max_value": 200,
        "output_name": "complex2d_traj0_prior_post_200ms_200timesteps.gif",
        "concat_prior_post": True,
    },
    "lorenz96_adaptive_0": {
        "dataset": "lorenz96",
        "image_directory": "save/lorenz96_pf_vis",
        "pattern": "sigma_y1.0_batch64_len500_pfN1000000_timestep*_42_0_adaptive.png",
        "frame_duration_ms": 400,
        "max_value": 64,
        "output_name": "lorenz96_zoomin_400ms_64timesteps.gif",
        "concat_prior_post": False,
    },
    "lorenz63_adaptive_b0": {
        "dataset": "lorenz63",
        "image_directory": "save/lorenz63_pf_vis",
        "pattern": "sigma_y1.0_batch64_len100_pfN1000000_timestep*_42_b0_POST_0_adaptive.png",
        "frame_duration_ms": 250,
        "max_value": 100,
        "output_name": "lorenz63_adaptive_b0_250ms_100timesteps.gif",
        "concat_prior_post": False,
    },
    "lorenz63_square_multiview_g0": {
        "dataset": "lorenz63",
        "image_directory": "save/lorenz63_pf_vis_square",
        "pattern": "sigma_y*_batch64_len200_pfN1000000_timestep*_42_g0_0_post_adaptive_xy.png",
        "frame_duration_ms": 250,
        "max_value": 200,
        "output_name": "lorenz63_square_g0_prior_post_xy_yz_xz_250ms_200timesteps.gif",
        "multi_view_l63": True,
        "plot_swd_vs_step": True,
        "distance_pt_pattern": "sigma_y*_batch64_len200_pfN1000000_42_non_gaussian_distance_detail.pt",
        "swd_output_name": "lorenz63_square_swd_vs_step.png",
        "concat_prior_post": False,
    },
    "lorenz63_arctan_multiview_g0": {
        "dataset": "lorenz63",
        "image_directory": "save/lorenz63_pf_vis_arctan",
        "pattern": "sigma_y*_batch64_len200_pfN1000000_timestep*_42_g0_0_post_adaptive_xy.png",
        "frame_duration_ms": 250,
        "max_value": 200,
        "output_name": "lorenz63_arctan_g0_prior_post_xy_yz_xz_250ms_200timesteps.gif",
        "multi_view_l63": True,
        "plot_swd_vs_step": True,
        "distance_pt_pattern": "sigma_y*_batch64_len200_pfN1000000_42_non_gaussian_distance_detail.pt",
        "swd_output_name": "lorenz63_arctan_swd_vs_step.png",
        "concat_prior_post": False,
    },
    "lorenz63_identity_multiview_g0": {
        "dataset": "lorenz63",
        "image_directory": "save/lorenz63_pf_vis",
        "pattern": "sigma_y*_batch64_len200_pfN1000000_timestep*_42_g0_0_post_adaptive_xy.png",
        "frame_duration_ms": 250,
        "max_value": 200,
        "output_name": "lorenz63_identity_g0_prior_post_xy_yz_xz_250ms_200timesteps.gif",
        "multi_view_l63": True,
        "plot_swd_vs_step": True,
        "distance_pt_pattern": "sigma_y*_batch64_len200_pfN1000000_42_non_gaussian_distance_detail.pt",
        "swd_output_name": "lorenz63_identity_swd_vs_step.png",
        "concat_prior_post": False,
    },
    "rossler_adaptive_0": {
        "dataset": "rossler",
        "image_directory": "save/rossler_pf_vis",
        "pattern": "sigma_y1.0_batch64_len500_pfN1000000_timestep*_42_0_adaptive.png",
        "frame_duration_ms": 400,
        "max_value": 500,
        "output_name": "rossler_adaptive_0_400ms_500timesteps.gif",
        "concat_prior_post": False,
    },
}

print("Available animation jobs:")
for key, cfg in ANIMATION_JOBS.items():
    print(f"- {key} ({cfg['dataset']})")



Available animation jobs:
- doubling1d_ring_g0_post (doubling1d)
- doubling1d_ring_g0_prior_post (doubling1d)
- complex2d_ring_g0_post (complex2d)
- complex2d_ring_g0_prior_post (complex2d)
- lorenz96_adaptive_0 (lorenz96)
- lorenz63_adaptive_b0 (lorenz63)
- lorenz63_square_multiview_g0 (lorenz63)
- lorenz63_arctan_multiview_g0 (lorenz63)
- lorenz63_identity_multiview_g0 (lorenz63)
- rossler_adaptive_0 (rossler)


In [4]:
# Select and run one or multiple jobs
SELECTED_JOBS = [
    "lorenz63_square_multiview_g0",
    "lorenz63_arctan_multiview_g0",
    "lorenz63_identity_multiview_g0",
    # "doubling1d_ring_g0_post",
    # "complex2d_ring_g0_post",
    # "doubling1d_ring_g0_prior_post",
    # "complex2d_ring_g0_prior_post",
    # "lorenz96_adaptive_0",
    # "lorenz63_adaptive_b0",
    # "rossler_adaptive_0",
]

RING_DATASETS = {"doubling1d", "complex2d"}
SLOW_SWITCH_MAX_STEPS = 30
SLOW_SWITCH_FRAME_MS = 600


def _replace_mode_token(text: str, target_mode: str) -> Optional[str]:
    """
    Replace sigma-range mode token in a pattern/name.
    Returns None if no adaptive/fixed token is found.
    """
    if "_adaptive" in text:
        return text.replace("_adaptive", f"_{target_mode}")
    if "_fixed" in text:
        return text.replace("_fixed", f"_{target_mode}")
    if "adaptive" in text:
        return text.replace("adaptive", target_mode)
    if "fixed" in text:
        return text.replace("fixed", target_mode)
    return None


def _apply_mode_to_output_name(name: str, target_mode: str) -> str:
    stem, ext = os.path.splitext(name)
    replaced_stem = _replace_mode_token(stem, target_mode)
    if replaced_stem is None:
        replaced_stem = f"{stem}_{target_mode}"
    return f"{replaced_stem}{ext}"


def derive_fixed_adaptive_jobs(job_key: str, cfg: dict):
    """
    For jobs whose pattern includes adaptive/fixed tokens, derive both modes.
    """
    pattern = cfg.get("pattern", "")
    adaptive_pattern = _replace_mode_token(pattern, "adaptive")
    fixed_pattern = _replace_mode_token(pattern, "fixed")

    if adaptive_pattern is None or fixed_pattern is None:
        return [(job_key, cfg)]

    derived = []
    for mode, mode_pattern in [("adaptive", adaptive_pattern), ("fixed", fixed_pattern)]:
        derived_cfg = dict(cfg)
        derived_cfg["pattern"] = mode_pattern
        derived_cfg["output_name"] = _apply_mode_to_output_name(cfg["output_name"], mode)

        if "swd_output_name" in derived_cfg:
            derived_cfg["swd_output_name"] = _apply_mode_to_output_name(
                derived_cfg["swd_output_name"],
                mode,
            )

        derived.append((f"{job_key}_{mode}", derived_cfg))

    return derived


def derive_ring_stat_jobs(job_key: str, cfg: dict):
    """
    For ring-map jobs on doubling1d/complex2d, also build CDF/PDF GIF jobs.

    Example transformation:
      pattern: ..._ring.png -> ..._ring_cdf.png / ..._ring_pdf.png
      output : xxx.gif      -> xxx_cdf.gif      / xxx_pdf.gif
    """
    dataset = cfg.get("dataset", "")
    pattern = cfg.get("pattern", "")
    if dataset not in RING_DATASETS:
        return []
    if not pattern.endswith("_ring.png"):
        return []

    derived = []
    for stat_name in ["cdf", "pdf"]:
        derived_cfg = dict(cfg)
        derived_cfg["pattern"] = pattern.replace("_ring.png", f"_ring_{stat_name}.png")

        stem, ext = os.path.splitext(cfg["output_name"])
        if ext == "":
            ext = ".gif"
        derived_cfg["output_name"] = f"{stem}_{stat_name}{ext}"

        derived.append((f"{job_key}_{stat_name}", derived_cfg))
    return derived


def derive_ring_switch_jobs(job_key: str, cfg: dict):
    """
    For doubling1d/complex2d ring jobs, create extra slow PRIOR->POST switching GIFs.
    - limit to <= 30 steps
    - generate both with-history and no-history versions
    """
    dataset = cfg.get("dataset", "")
    pattern = cfg.get("pattern", "")

    if dataset not in RING_DATASETS:
        return []
    if cfg.get("concat_prior_post", False):
        return []
    if not pattern.endswith("_ring.png"):
        return []
    if "POST" not in pattern:
        return []

    max_value = cfg.get("max_value")
    if isinstance(max_value, int):
        switch_steps = min(SLOW_SWITCH_MAX_STEPS, max_value)
    else:
        switch_steps = SLOW_SWITCH_MAX_STEPS

    frame_duration_ms = max(int(cfg.get("frame_duration_ms", 200)), SLOW_SWITCH_FRAME_MS)

    stem, ext = os.path.splitext(cfg["output_name"])
    if ext == "":
        ext = ".gif"

    hist_cfg = dict(cfg)
    hist_cfg["switch_prior_post"] = True
    hist_cfg["switch_left_tag"] = "PRIOR"
    hist_cfg["switch_right_tag"] = "POST"
    hist_cfg["switch_max_steps"] = switch_steps
    hist_cfg["frame_duration_ms"] = frame_duration_ms
    hist_cfg["concat_prior_post"] = False
    hist_cfg["output_name"] = f"{stem}_switch_prior_post_slow_hist{ext}"

    nohist_cfg = dict(hist_cfg)
    nohist_cfg["pattern"] = pattern.replace("POST", "POST_nohist", 1)
    nohist_cfg["switch_left_tag"] = "PRIOR_nohist"
    nohist_cfg["switch_right_tag"] = "POST_nohist"
    nohist_cfg["output_name"] = f"{stem}_switch_prior_post_slow_nohist{ext}"

    return [
        (f"{job_key}_switch_hist", hist_cfg),
        (f"{job_key}_switch_nohist", nohist_cfg),
    ]


expanded_jobs = []
for job_key in SELECTED_JOBS:
    cfg = ANIMATION_JOBS[job_key]

    mode_jobs = derive_fixed_adaptive_jobs(job_key, cfg)
    for mode_job_key, mode_cfg in mode_jobs:
        expanded_jobs.append((mode_job_key, mode_cfg))
        expanded_jobs.extend(derive_ring_stat_jobs(mode_job_key, mode_cfg))
        expanded_jobs.extend(derive_ring_switch_jobs(mode_job_key, mode_cfg))

for job_key, cfg in expanded_jobs:
    image_directory = cfg["image_directory"]
    pattern = cfg["pattern"]
    frame_duration_ms = cfg["frame_duration_ms"]
    max_value = cfg["max_value"]
    output_gif_file = os.path.join(image_directory, cfg["output_name"])

    if cfg.get("multi_view_l63", False):
        print(f"[{job_key}] building L63 multiview prior/post GIF from pattern: {pattern}")
        create_l63_prior_post_projection_gif(
            image_directory=image_directory,
            post_xy_pattern=pattern,
            gif_path=output_gif_file,
            duration=frame_duration_ms,
            max_value=max_value,
            sigma_value=cfg.get("sigma_value"),
        )
        if cfg.get("plot_swd_vs_step", False):
            plot_swd_vs_step(
                image_directory=image_directory,
                distance_pt_pattern=cfg.get(
                    "distance_pt_pattern",
                    "sigma_y*_batch64_len200_pfN1000000_42_non_gaussian_distance_detail.pt",
                ),
                output_name=cfg.get("swd_output_name"),
            )
        continue

    if cfg.get("switch_prior_post", False):
        print(
            f"[{job_key}] building slow PRIOR->POST switch GIF from pattern: {pattern} "
            f"(max_steps={cfg.get('switch_max_steps')})"
        )
        create_prior_post_switch_gif(
            image_directory=image_directory,
            pattern=pattern,
            gif_path=output_gif_file,
            duration=frame_duration_ms,
            max_steps=cfg.get("switch_max_steps", SLOW_SWITCH_MAX_STEPS),
            left_tag=cfg.get("switch_left_tag", "PRIOR"),
            right_tag=cfg.get("switch_right_tag", "POST"),
        )
        continue

    gif_pattern = pattern
    if cfg.get("concat_prior_post", False):
        created, _ = pair_prior_post(
            image_directory=image_directory,
            pattern=pattern,
        )
        print(f"[{job_key}] concatenated PRIOR/POST frames: {created}")
        gif_pattern = pattern.replace("POST", "PRIOR_POST", 1)

    print(f"[{job_key}] building GIF from pattern: {gif_pattern}")
    create_gif_robust(
        image_folder=image_directory,
        file_pattern=gif_pattern,
        gif_path=output_gif_file,
        duration=frame_duration_ms,
        max_value=max_value,
    )



[lorenz63_square_multiview_g0_adaptive] building L63 multiview prior/post GIF from pattern: sigma_y*_batch64_len200_pfN1000000_timestep*_42_g0_0_post_adaptive_xy.png
Saved frame snapshot: save/lorenz63_pf_vis_square/lorenz63_square_g0_prior_post_xy_yz_xz_250ms_200timesteps_adaptive_step25pct_timestep50.png (percentile=25%, timestep=50)
Saved frame snapshot: save/lorenz63_pf_vis_square/lorenz63_square_g0_prior_post_xy_yz_xz_250ms_200timesteps_adaptive_step50pct_timestep100.png (percentile=50%, timestep=100)
Saved frame snapshot: save/lorenz63_pf_vis_square/lorenz63_square_g0_prior_post_xy_yz_xz_250ms_200timesteps_adaptive_step75pct_timestep150.png (percentile=75%, timestep=150)
Saved frame snapshot: save/lorenz63_pf_vis_square/lorenz63_square_g0_prior_post_xy_yz_xz_250ms_200timesteps_adaptive_step100pct_timestep199.png (percentile=100%, timestep=199)
L63 multiview GIF saved at: save/lorenz63_pf_vis_square/lorenz63_square_g0_prior_post_xy_yz_xz_250ms_200timesteps_adaptive.gif (sigma_y=16

/tmp/ipykernel_101585/3537500570.py:513: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(distance_pt_path, map_location="cpu")


SWD-vs-step figure saved at: save/lorenz63_pf_vis_square/lorenz63_square_swd_vs_step_adaptive.png
[lorenz63_square_multiview_g0_fixed] building L63 multiview prior/post GIF from pattern: sigma_y*_batch64_len200_pfN1000000_timestep*_42_g0_0_post_fixed_xy.png
Saved frame snapshot: save/lorenz63_pf_vis_square/lorenz63_square_g0_prior_post_xy_yz_xz_250ms_200timesteps_fixed_step25pct_timestep50.png (percentile=25%, timestep=50)
Saved frame snapshot: save/lorenz63_pf_vis_square/lorenz63_square_g0_prior_post_xy_yz_xz_250ms_200timesteps_fixed_step50pct_timestep100.png (percentile=50%, timestep=100)
Saved frame snapshot: save/lorenz63_pf_vis_square/lorenz63_square_g0_prior_post_xy_yz_xz_250ms_200timesteps_fixed_step75pct_timestep150.png (percentile=75%, timestep=150)
Saved frame snapshot: save/lorenz63_pf_vis_square/lorenz63_square_g0_prior_post_xy_yz_xz_250ms_200timesteps_fixed_step100pct_timestep199.png (percentile=100%, timestep=199)
L63 multiview GIF saved at: save/lorenz63_pf_vis_square/lo

In [ ]:
# Optional helper: quickly inspect jobs grouped by dataset
def list_jobs_by_dataset(dataset: str):
    return [k for k, v in ANIMATION_JOBS.items() if v["dataset"] == dataset]

for ds in ["doubling1d", "complex2d", "lorenz63", "lorenz96", "rossler"]:
    print(ds, "->", list_jobs_by_dataset(ds))



doubling1d -> ['doubling1d_ring_g0_post', 'doubling1d_ring_g0_prior_post']
complex2d -> ['complex2d_ring_g0_post', 'complex2d_ring_g0_prior_post']
lorenz63 -> ['lorenz63_adaptive_b0', 'lorenz63_square_multiview_g0', 'lorenz63_arctan_multiview_g0', 'lorenz63_identity_multiview_g0']
lorenz96 -> ['lorenz96_adaptive_0']
rossler -> ['rossler_adaptive_0']


: 